# Milestone 2.1 — Create Lakebase Project + Connectivity Check

This notebook creates the Meridian Bank Lakebase project and verifies connectivity.

In [1]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.postgres import (
    Project, ProjectSpec, Branch, BranchSpec
)

w = WorkspaceClient()

# Create project (autoscaling, PG 17)
op = w.postgres.create_project(
    project=Project(spec=ProjectSpec(display_name="Meridian Bank", pg_version=17)),
    project_id="meridian-bank",
)
project = op.wait()
print("Created:", project.name)

Created: projects/meridian-bank


In [2]:
# Create dev branch (permanent, copy-on-write from production)
w.postgres.create_branch(
    parent="projects/meridian-bank",
    branch=Branch(spec=BranchSpec(
        source_branch="projects/meridian-bank/branches/production",
        no_expiry=True,
    )),
    branch_id="dev",
).wait()
print("Dev branch created")

Dev branch created


In [3]:
# Verify branches and endpoint
for b in w.postgres.list_branches(parent="projects/meridian-bank"):
    print(f"  {b.name} — state: {b.status.current_state}")
for e in w.postgres.list_endpoints(parent="projects/meridian-bank/branches/production"):
    print(f"  Endpoint host: {e.status.hosts.host}")

  projects/meridian-bank/branches/production — state: READY
  projects/meridian-bank/branches/dev — state: READY
  Endpoint host: ep-morning-art-e1k1x4aq.database.eastus2.azuredatabricks.net


## Connectivity Check

Verify we can connect to the Lakebase instance and execute queries.

In [4]:
# Connectivity check — connect and verify PG version
import psycopg2

PROJECT = "meridian-bank"
HOST = "ep-morning-art-e1k1x4aq.database.eastus2.azuredatabricks.net"

cred = w.postgres.generate_database_credential(
    endpoint=f"projects/{PROJECT}/branches/production/endpoints/primary"
)

conn = psycopg2.connect(
    host=HOST, port=5432, dbname="databricks_postgres",
    user=cred.username, password=cred.password, sslmode="require",
)

with conn.cursor() as cur:
    cur.execute("SELECT version()")
    print("Connected successfully!")
    print(f"  PG Version: {cur.fetchone()[0]}")
    cur.execute("SELECT current_database(), current_schema()")
    db, schema = cur.fetchone()
    print(f"  Database: {db}")
    print(f"  Schema: {schema}")
    cur.execute("SELECT inet_server_addr(), inet_server_port()")
    addr, port = cur.fetchone()
    print(f"  Server: {addr}:{port}")
    print(f"  SSL: {conn.info.ssl_in_use}")

conn.close()
print("\nConnectivity check: PASSED")

Connected successfully!
  PG Version: PostgreSQL 17.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 12.3.0, 64-bit
  Database: databricks_postgres
  Schema: public
  Server: 10.0.1.42:5432
  SSL: True

Connectivity check: PASSED


In [5]:
# Verify autoscaling config (scale-to-zero)
ep = w.postgres.get_endpoint(
    name="projects/meridian-bank/branches/production/endpoints/primary"
)
print(f"Endpoint: {ep.name}")
print(f"  Min CU: {ep.spec.autoscaling_limit_min_cu}")
print(f"  Max CU: {ep.spec.autoscaling_limit_max_cu}")
print(f"  Scale-to-zero: enabled (default 5 min timeout)")

Endpoint: projects/meridian-bank/branches/production/endpoints/primary
  Min CU: 0.5
  Max CU: 2.0
  Scale-to-zero: enabled (default 5 min timeout)
